In [0]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')
import time

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark Connect").getOrCreate()
from pyspark.sql.types import IntegerType
import pyspark.sql.functions as F

In [0]:
# spark.conf.set("spark.sql.shuffle.partitions", "3")
spark.conf.get("spark.sql.adaptive.enabled")
#spark.conf.set("spark.sql.adaptive.enabled", "false")


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8376543825036194>, line 2
      1 # spark.conf.set("spark.sql.shuffle.partitions", "3")
----> 2 spark.conf.get("spark.sql.adaptive.enabled")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:80, in RuntimeConf.get(self, key, default)
     76     op_get_with_default = proto.ConfigRequest.GetWithDefault(
     77         pairs=[proto.KeyValue(key=key, value=cast(Optional[str], default))]
     78     )
     79     operation = proto.ConfigRequest.Operation(get_with_default=op_get_with_default)
---> 80 result = self._client.config(operation)
     81 return result.pairs[0][1]

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:2246, in SparkConnectClient.config(self, operation)
   2244     raise SparkConnectException("Invalid state during retry exception handlin

In [0]:
transactions_file = "/Volumes/workspace/pyspark/volumes/data_skew/transactions.parquet"
customer_file = "/Volumes/workspace/pyspark/volumes/data_skew/customers.parquet/"

df_transactions = spark.read.parquet(transactions_file)
df_customers = spark.read.parquet(customer_file)

In [0]:
df_transactions.printSchema()
df_transactions.show(5, False)

root
 |-- cust_id: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- txn_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- year: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day: string (nullable = true)
 |-- expense_type: string (nullable = true)
 |-- amt: string (nullable = true)
 |-- city: string (nullable = true)

+----------+----------+----------+---------------+----------+----+-----+---+-------------+------+-----------+
|cust_id   |start_date|end_date  |txn_id         |date      |year|month|day|expense_type |amt   |city       |
+----------+----------+----------+---------------+----------+----+-----+---+-------------+------+-----------+
|C0YDPQWPBJ|2010-07-01|2018-12-01|TZ5SMKZY9S03OQJ|2018-10-07|2018|10   |7  |Entertainment|10.42 |boston     |
|C0YDPQWPBJ|2010-07-01|2018-12-01|TYIAPPNU066CJ5R|2016-03-27|2016|3    |27 |Motor/Travel |44.34 |portland   |
|C0YDPQWPBJ|2010-07-01|201

In [0]:
df_customers.printSchema()
df_customers.show(5, False)

root
 |-- cust_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthday: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- city: string (nullable = true)

+----------+-------------+---+------+----------+-----+-----------+
|cust_id   |name         |age|gender|birthday  |zip  |city       |
+----------+-------------+---+------+----------+-----+-----------+
|C007YEYTX9|Aaron Abbott |34 |Female|7/13/1991 |97823|boston     |
|C00B971T1J|Aaron Austin |37 |Female|12/16/2004|30332|chicago    |
|C00WRSJF1Q|Aaron Barnes |29 |Female|3/11/1977 |23451|denver     |
|C01AZWQMF3|Aaron Barrett|31 |Male  |7/9/1998  |46613|los_angeles|
|C01BKUFRHA|Aaron Becker |54 |Male  |11/24/1979|40284|san_diego  |
+----------+-------------+---+------+----------+-----+-----------+
only showing top 5 rows


In [0]:
(
    df_transactions
    .groupBy("cust_id")
    .agg(F.countDistinct("txn_id").alias("ct"))
    .orderBy(F.desc("ct"))
    .show(5, False)
)

+----------+--------+
|cust_id   |ct      |
+----------+--------+
|C0YDPQWPBJ|17539732|
|C3KUDEN3KO|7999    |
|CBW3FMEAU7|7999    |
|C89FCEGPJP|7999    |
|CHNFNR89ZV|7998    |
+----------+--------+
only showing top 5 rows


In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8376543825036199>, line 1
----> 1 spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:46, in RuntimeConf.set(self, key, value)
     44 op_set = proto.ConfigRequest.Set(pairs=[proto.KeyValue(key=key, value=value)])
     45 operation = proto.ConfigRequest.Operation(set=op_set)
---> 46 result = self._client.config(operation)
     47 for warn in result.warnings:
     48     warnings.warn(warn)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:2246, in SparkConnectClient.config(self, operation)
   2244     raise SparkConnectException("Invalid state during retry exception handling.")
   2245 except Exception as error:
-> 2246     self._handle_error(error)

File /databricks/python/lib/python3.12/site-pac

In [0]:
df_txn_details = (
    df_transactions.join(
        df_customers,
        on="cust_id",
        how="inner"
    )
)

In [0]:
start_time = time.time()
df_txn_details.count()
print(f"time taken: {time.time() - start_time}")

39790092

time taken: 1.0809423923492432


In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewedJoin.enabled", "true")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8376543825036202>, line 1
----> 1 spark.conf.set("spark.sql.adaptive.enabled", "true")
      2 spark.conf.set("spark.sql.adaptive.skewedJoin.enabled", "true")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:46, in RuntimeConf.set(self, key, value)
     44 op_set = proto.ConfigRequest.Set(pairs=[proto.KeyValue(key=key, value=value)])
     45 operation = proto.ConfigRequest.Operation(set=op_set)
---> 46 result = self._client.config(operation)
     47 for warn in result.warnings:
     48     warnings.warn(warn)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:2246, in SparkConnectClient.config(self, operation)
   2244     raise SparkConnectException("Invalid state during retry exception handling.")
   2245 except Exception as error:
-> 2246     self._ha

In [0]:
df_txn_details = (
    df_transactions.join(
        df_customers,
        on="cust_id",
        how="inner"
    )
)

In [0]:
start_time = time.time()
df_txn_details.count()
print(f"time taken: {time.time() - start_time}")

39790092

time taken: 0.8151376247406006


In [0]:
# 10MB = 10485760 Bytes
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8376543825036205>, line 2
      1 # 10MB = 10485760 Bytes
----> 2 spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:46, in RuntimeConf.set(self, key, value)
     44 op_set = proto.ConfigRequest.Set(pairs=[proto.KeyValue(key=key, value=value)])
     45 operation = proto.ConfigRequest.Operation(set=op_set)
---> 46 result = self._client.config(operation)
     47 for warn in result.warnings:
     48     warnings.warn(warn)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:2246, in SparkConnectClient.config(self, operation)
   2244     raise SparkConnectException("Invalid state during retry exception handling.")
   2245 except Exception as error:
-> 2246     self._handle_error(error)

File /dat

In [0]:
df_txn_details = (
    df_transactions.join(
        F.broadcast(df_customers),
        on="cust_id",
        how="inner"
    )
)

In [0]:
start_time = time.time()
df_txn_details.count()
print(f"time taken: {time.time() - start_time}")

39790092

time taken: 0.8017888069152832
